# Bayesian DOE: Multi-Objective QD Synthesis Optimization

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/samdhole/materials-analysis/blob/main/multiobjective_bo_qd_synthesis.ipynb)
&nbsp;
[![GitHub](https://img.shields.io/badge/GitHub-View_Source-181717?logo=github)](https://github.com/samdhole/materials-analysis/blob/main/multiobjective_bo_qd_synthesis.ipynb)

---

## What this notebook does

This notebook runs a multi-objective Bayesian optimization campaign for
Mn-doped lead-halide perovskite QD synthesis. Given 5 synthesis parameters
(2 categorical, 3 continuous), it recommends which 5 conditions to run next
after each batch of measurements, steering the search toward a Pareto front
across three competing optical targets.

**Use it two ways:**
- **Demo mode** — run top-to-bottom with the included example data to see
  the full campaign structure and Pareto visualization without touching a lab.
- **Real campaign mode** — run each "recommend" cell, synthesize in your lab,
  then fill in your measured values using the form cells below each round.

---

**Domain:** Mn-doped mixed-halide QD-polymer composites for scintillator and
luminescence applications. Parameters match synthesis work on QD scintillators
for DOE-funded dual-readout calorimetry detector programs.

**Synthesis route:** Mechano-chemical ball milling — 30–45 min per 3-composition
batch vs. ~1 day by hot-injection. Enables rapid iterative DOE.

**Optimization targets (all competing):**
- Stokes shift ↑ (emission–absorption gap; minimizes self-absorption in detector films)
- Quantum yield ↑ (fraction of absorbed photons re-emitted as light)
- Absorption edge ↓ (shorter wavelength = better Cherenkov capture)

**Stack:** NumPy · SciPy · pandas · BayBE · BoTorch · Plotly

In [1]:
#@title ⚙️ Install dependencies and import libraries { display-mode: "form" }
#@markdown Run this cell first. Takes ~60 s on first run (installs BayBE + BoTorch).

import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "baybe", "--quiet"],
               capture_output=True)

import io
import warnings
import numpy as np
import pandas as pd
import plotly.express as px
from IPython.display import display, HTML
from scipy.stats.qmc import LatinHypercube, scale
from baybe.campaign import Campaign
from baybe.parameters import CategoricalParameter, NumericalContinuousParameter
from baybe.targets import NumericalTarget
from baybe.objectives import ParetoObjective
from baybe.recommenders.pure.bayesian import BotorchRecommender
from baybe.acquisition import qLogNParEGO
from baybe.searchspace import SearchSpace

try:
    from linear_operator.utils.warnings import NumericalWarning
    warnings.filterwarnings("ignore", category=NumericalWarning)
except ImportError:
    pass
try:
    from baybe.utils.dataframe import InputDataTypeWarning
    warnings.filterwarnings("ignore", category=InputDataTypeWarning)
except ImportError:
    pass
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

print("✓ Libraries loaded.")

✓ Libraries loaded.


---
## 1 · Parameter Space and Initial Design

### The 5 synthesis parameters

| Parameter | Type | Range / Options | What it controls |
|-----------|------|-----------------|-----------------|
| Capping Ligand | Categorical | Oleylamine, Octylamine, APTES | Surface coating; governs size, solubility, surface defects |
| A-Site Cation | Categorical | Cs, PEA, MA | ABX₃ crystal site; governs symmetry, stability, emission lifetime |
| Mn²⁺ Doping | Continuous | 1–50 mol% | Energy-transfer dopant; red-shifts emission, enlarges Stokes shift |
| X Composition | Continuous | 0–1 (Cl:Br ratio) | Halide ratio; tunes bandgap and absorption/emission wavelength |
| Milling Time | Continuous | 3–20 min | Particle size and reaction yield |

**Latin Hypercube Sampling (LHS)** divides each parameter's range into equal
intervals and places exactly one sample per interval — guaranteeing uniform
coverage with 15 experiments instead of hundreds for full factorial.

In [2]:
#@title 🔷 Generate 15 LHS initial conditions
LIGANDS = ["Oleylamine", "Octylamine", "APTES"]
CATIONS = ["Cs", "PEA", "MA"]

lower = [0,    0,    1.0, 0.0,  3.0]
upper = [2.99, 2.99, 50.0, 1.0, 20.0]

sampler    = LatinHypercube(d=5, seed=42)
lhs_scaled = scale(sampler.random(n=15), lower, upper)

df_lhs = pd.DataFrame(lhs_scaled,
                      columns=["Capping Ligand", "A-Site Cation",
                                "Mn2+ Doping", "X Composition", "Milling Time"])
df_lhs["Capping Ligand"] = df_lhs["Capping Ligand"].apply(lambda x: LIGANDS[int(x)])
df_lhs["A-Site Cation"]  = df_lhs["A-Site Cation"].apply(lambda x: CATIONS[int(x)])
df_lhs = df_lhs.round(2)

print("LHS initial conditions (15 experiments):")
display(df_lhs.style
        .set_caption("Space-filling initial design — synthesize all 15 before starting the loop")
        .format({"Mn2+ Doping": "{:.2f}", "X Composition": "{:.2f}", "Milling Time": "{:.2f}"}))

LHS initial conditions (15 experiments):


,Capping Ligand,A-Site Cation,Mn2+ Doping,X Composition,Milling Time
0,Oleylamine,MA,24.33,0.89,4.03
1,Oleylamine,Cs,31.10,0.52,10.42
2,Octylamine,MA,21.76,0.08,12.70
3,APTES,Cs,39.99,0.74,6.82
4,APTES,PEA,17.43,0.61,17.98
5,Octylamine,PEA,36.79,0.72,14.69
6,Oleylamine,Cs,29.34,0.98,16.07
7,Octylamine,Cs,12.51,0.45,5.64
8,Oleylamine,PEA,41.18,0.25,8.86
9,APTES,Cs,9.86,0.55,11.91


---
## 2 · Initial Measurements

After synthesizing all 15 LHS conditions, measure **Stokes shift (nm)**,
**quantum yield (0.0–1.0 fraction)**, and **absorption edge (nm)** for each.

**Demo mode:** the table below contains example measurements. Replace with
your own data if running a real campaign.

In [3]:
#@title 📋 Enter initial 15 measurements { display-mode: "form" }
#@markdown **Demo data is pre-filled.** For a real campaign, edit the CSV string below
#@markdown (keep the same column order) and re-run.

_initial_csv = """\
Capping Ligand,A-Site Cation,Mn2+ Doping,X Composition,Milling Time,Stokes Shift,Quantum Yield,Absorption Edge
APTES,MA,32.03,0.12,4.66,178,0.21,398
Octylamine,MA,28.07,0.20,3.35,182,0.29,401
Oleylamine,PEA,19.28,0.16,12.22,193,0.58,432
APTES,PEA,37.71,0.04,16.25,212,0.89,438
Oleylamine,Cs,2.11,0.43,11.67,208,0.85,419
Oleylamine,MA,25.32,0.55,13.28,175,0.31,415
Oleylamine,PEA,48.94,0.48,6.51,185,0.42,441
APTES,MA,10.34,0.82,8.43,201,0.72,425
Octylamine,MA,42.26,0.61,6.27,169,0.15,418
Oleylamine,PEA,6.52,0.31,10.58,171,0.25,435
Octylamine,Cs,23.30,0.99,8.82,181,0.33,433
APTES,Cs,35.08,0.37,19.01,173,0.18,428
Octylamine,Cs,43.53,0.67,17.39,165,0.11,430
Octylamine,PEA,14.71,0.88,14.90,188,0.45,445
APTES,Cs,13.68,0.75,18.72,195,0.56,422"""

df_initial = pd.read_csv(io.StringIO(_initial_csv))
target_cols = ["Stokes Shift", "Quantum Yield", "Absorption Edge"]
df_initial[target_cols] = df_initial[target_cols].astype(float)

print(f"✓ {len(df_initial)} initial measurements loaded.\n")
print("Summary statistics:")
display(df_initial[target_cols].describe().round(3).style.set_caption("Initial 15 experiments"))

✓ 15 initial measurements loaded.

Summary statistics:


,Stokes Shift,Quantum Yield,Absorption Edge
count,15.000000,15.000000,15.000000
mean,185.067000,0.420000,425.333000
std,14.285000,0.251000,13.547000
min,165.000000,0.110000,398.000000
25%,174.000000,0.230000,418.500000
50%,182.000000,0.330000,428.000000
75%,194.000000,0.570000,434.000000
max,212.000000,0.890000,445.000000


---
## 3 · Campaign Setup

Key design decisions:

- **`SearchSpace.from_product`** — constructs the full parameter space from the
  Cartesian product of all categorical and continuous parameters. Required for
  mixed spaces; base recommenders don't handle categoricals.
- **`ParetoObjective`** — returns a batch on the Pareto front rather than
  collapsing to a scalar; no objective weighting required.
- **`qLogNParEGO`** acquisition — log-space Noisy Pareto Expected Improvement.
  Batch-aware, handles observation noise, scales better than q-NEHVI for
  larger Pareto fronts. Recommended over EHI for batch_size > 1.

In [4]:
#@title ⚙️ Initialize campaign
parameters = [
    CategoricalParameter(name="Capping Ligand", values=LIGANDS),
    CategoricalParameter(name="A-Site Cation",  values=CATIONS),
    NumericalContinuousParameter(name="Mn2+ Doping",   bounds=[1.0,  50.0]),
    NumericalContinuousParameter(name="X Composition", bounds=[0.0,   1.0]),
    NumericalContinuousParameter(name="Milling Time",  bounds=[3.0,  20.0]),
]

targets = [
    NumericalTarget(name="Stokes Shift",    mode="MAX"),
    NumericalTarget(name="Quantum Yield",   mode="MAX"),
    NumericalTarget(name="Absorption Edge", mode="MIN"),
]

campaign = Campaign(
    searchspace=SearchSpace.from_product(parameters=parameters),
    objective=ParetoObjective(targets=targets),
    recommender=BotorchRecommender(acquisition_function=qLogNParEGO()),
)

campaign.add_measurements(df_initial)
print(f"✓ Campaign initialized. {len(campaign.measurements)} measurements loaded.")
print("  Acquisition function: qLogNParEGO (log-space Noisy Pareto Expected Improvement)")
print("  Search space: 3×3 categorical × 3 continuous = effectively 12-dimensional")

✓ Campaign initialized. 15 measurements loaded.
  Acquisition function: qLogNParEGO (log-space Noisy Pareto Expected Improvement)
  Search space: 3×3 categorical × 3 continuous = effectively 12-dimensional


---
## 4 · Iterative Optimization Loop

### How to use this section (real campaign mode)

For each round:
1. **Run the "Get Recommendations" cell** → note the 5 suggested conditions
2. **Synthesize** those conditions in the ball mill (~30–45 min each)
3. **Measure** Stokes shift, quantum yield, and absorption edge
4. **Fill in the "Enter Results" form cell** with your values, then run it
5. **Proceed to the next round**

The GP surrogate updates after every batch. Early rounds explore broadly
(cold-start uncertainty is high); later rounds concentrate on the Pareto
front as the model accumulates data.

---

In [5]:
#@title 🔷 Round 1 — Get Recommendations
print("── Round 1 Recommendations ──────────────────────────────────────────")
print("Surrogate trained on 15 LHS points. Early-round uncertainty is high;")
print("recommendations explore more broadly than later rounds.\n")
r1_rec = campaign.recommend(batch_size=5)
display(r1_rec.reset_index(drop=True).style.set_caption(
    "Round 1 — Synthesize these 5 conditions, then fill in the form below"))

── Round 1 Recommendations ──────────────────────────────────────────
Surrogate trained on 15 LHS points. Early-round uncertainty is high;
recommendations explore more broadly than later rounds.



,Capping Ligand,A-Site Cation,Milling Time,Mn2+ Doping,X Composition
0,Oleylamine,Cs,13.239738,2.110162,0.430152
1,Oleylamine,Cs,12.135817,2.110064,0.316321
2,Oleylamine,Cs,10.312170,2.113454,0.484657
3,Oleylamine,Cs,6.352948,3.908343,0.413452
4,APTES,MA,4.514491,49.356452,0.093723


In [6]:
#@title 🧪 Round 1 — Enter Your Measured Results { display-mode: "form" }
#@markdown After synthesizing the 5 conditions above, enter your measurements here.
#@markdown **Stokes Shift** (nm) · **Quantum Yield** (0.0–1.0 fraction) · **Absorption Edge** (nm)
#@markdown Separate values with commas, in the same order as the recommended conditions above.

stokes_r1   = "211, 205, 208, 206, 196"   #@param {type:"string"}
qy_r1       = "0.90, 0.85, 0.94, 0.89, 0.71"  #@param {type:"string"}
absedge_r1  = "415, 428, 411, 406, 424"   #@param {type:"string"}

def _parse_csv_row(s):
    return [float(v.strip()) for v in s.split(",")]

def _add_round(rec_df, stokes_s, qy_s, absedge_s, label):
    sv, qv, av = _parse_csv_row(stokes_s), _parse_csv_row(qy_s), _parse_csv_row(absedge_s)
    assert len(sv) == len(rec_df) == len(qv) == len(av), \
        f"Expected {len(rec_df)} values per measurement, got {len(sv)}/{len(qv)}/{len(av)}"
    df = rec_df.copy().reset_index(drop=True)
    df["Stokes Shift"]    = sv
    df["Quantum Yield"]   = qv
    df["Absorption Edge"] = av
    campaign.add_measurements(df)
    print(f"\n✓ {label} complete — total experiments: {len(campaign.measurements)}")
    print(f"   Stokes:   {min(sv):.0f}–{max(sv):.0f} nm")
    print(f"   QY:       {min(qv):.0%}–{max(qv):.0%}")
    print(f"   Abs edge: {min(av):.0f}–{max(av):.0f} nm")
    return df

r1_data = _add_round(r1_rec, stokes_r1, qy_r1, absedge_r1, "Round 1")


✓ Round 1 complete — total experiments: 20
   Stokes:   196–211 nm
   QY:       71%–94%
   Abs edge: 406–428 nm


In [7]:
#@title 🔷 Round 2 — Get Recommendations
print("── Round 2 Recommendations ──────────────────────────────────────────")
print("Surrogate now fitted on 20 points. APTES/MA + low Mn region starting")
print("to emerge as promising; recommendations become more targeted.\n")
r2_rec = campaign.recommend(batch_size=5)
display(r2_rec.reset_index(drop=True).style.set_caption(
    "Round 2 — Synthesize these 5 conditions, then fill in the form below"))

── Round 2 Recommendations ──────────────────────────────────────────
Surrogate now fitted on 20 points. APTES/MA + low Mn region starting
to emerge as promising; recommendations become more targeted.



,Capping Ligand,A-Site Cation,Milling Time,Mn2+ Doping,X Composition
0,Oleylamine,Cs,6.968430,3.881304,0.558000
1,Oleylamine,Cs,11.732781,2.352734,0.650197
2,Oleylamine,Cs,4.888024,5.275174,0.483552
3,Oleylamine,Cs,16.365849,2.686966,0.625580
4,Oleylamine,Cs,18.894659,3.323330,0.504078


In [8]:
#@title 🧪 Round 2 — Enter Your Measured Results { display-mode: "form" }
#@markdown After synthesizing the 5 conditions above, enter your measurements here.
#@markdown **Stokes Shift** (nm) · **Quantum Yield** (0.0–1.0 fraction) · **Absorption Edge** (nm)

stokes_r2   = "205, 201, 188, 208, 209"   #@param {type:"string"}
qy_r2       = "0.85, 0.81, 0.65, 0.84, 0.92"  #@param {type:"string"}
absedge_r2  = "412, 395, 426, 450, 403"   #@param {type:"string"}

r2_data = _add_round(r2_rec, stokes_r2, qy_r2, absedge_r2, "Round 2")


✓ Round 2 complete — total experiments: 25
   Stokes:   188–209 nm
   QY:       65%–92%
   Abs edge: 395–450 nm


In [9]:
#@title 🔷 Round 3 — Get Recommendations
print("── Round 3 Recommendations ──────────────────────────────────────────")
print("Surrogate fitted on 25 points. APTES/MA + low Mn confirmed as Pareto")
print("cluster; recommendations concentrate in the 395–410 nm absorption edge region.\n")
r3_rec = campaign.recommend(batch_size=5)
display(r3_rec.reset_index(drop=True).style.set_caption(
    "Round 3 — Synthesize these 5 conditions, then fill in the form below"))

── Round 3 Recommendations ──────────────────────────────────────────
Surrogate fitted on 25 points. APTES/MA + low Mn confirmed as Pareto
cluster; recommendations concentrate in the 395–410 nm absorption edge region.



,Capping Ligand,A-Site Cation,Milling Time,Mn2+ Doping,X Composition
0,Oleylamine,Cs,19.360909,2.957466,0.485693
1,Oleylamine,Cs,11.750405,3.098530,0.582586
2,Oleylamine,Cs,18.751563,2.788496,0.415646
3,Oleylamine,Cs,18.813771,3.055162,0.554348
4,Oleylamine,Cs,8.833567,3.024407,0.443047


In [10]:
#@title 🧪 Round 3 — Enter Your Measured Results { display-mode: "form" }
#@markdown After synthesizing the 5 conditions above, enter your measurements here.
#@markdown **Stokes Shift** (nm) · **Quantum Yield** (0.0–1.0 fraction) · **Absorption Edge** (nm)

stokes_r3   = "198, 203, 207, 209, 192"   #@param {type:"string"}
qy_r3       = "0.78, 0.83, 0.88, 0.91, 0.45"  #@param {type:"string"}
absedge_r3  = "416, 395, 397, 405, 399"   #@param {type:"string"}

r3_data = _add_round(r3_rec, stokes_r3, qy_r3, absedge_r3, "Round 3")


✓ Round 3 complete — total experiments: 30
   Stokes:   192–209 nm
   QY:       45%–91%
   Abs edge: 395–416 nm


In [11]:
#@title 🔷 Round 4 — Final Recommendations (no synthesis required)
#@markdown Round 4 generates the final suggested conditions based on all 30 experiments.
#@markdown These are the best starting points for any follow-on work.
print("── Round 4 Final Recommendations ────────────────────────────────────")
print(f"Surrogate fitted on {len(campaign.measurements)} experiments.\n")
r4_rec = campaign.recommend(batch_size=5)
display(r4_rec.reset_index(drop=True).style.set_caption(
    "Round 4 — Predicted optimal conditions (no synthesis needed for this demo)"))

── Round 4 Final Recommendations ────────────────────────────────────
Surrogate fitted on 30 experiments.



,Capping Ligand,A-Site Cation,Milling Time,Mn2+ Doping,X Composition
0,Oleylamine,Cs,18.549194,5.127928,0.422419
1,Oleylamine,Cs,18.517733,1.000000,0.412301
2,Oleylamine,Cs,18.575713,5.500134,0.250163
3,Oleylamine,Cs,11.840640,5.100074,0.605804
4,Oleylamine,Cs,11.775769,11.514355,0.707924


---
## 5 · Pareto Front

Non-dominated solutions across all 30 experiments. A condition is
**Pareto-optimal** if no other experiment beats it on **all three** objectives
simultaneously.

- Red diamonds = Pareto-optimal conditions (worth running at scale)
- Gray circles = dominated experiments (something else beats them on every axis)

The front is the boundary of achievable tradeoffs — each point represents
a condition where any further improvement on one objective requires sacrificing
another.

In [12]:
#@title 📊 Compute Pareto front and visualize { display-mode: "form" }
all_data = campaign.measurements.copy()

def pareto_front(df, objectives):
    vals      = df[[o[0] for o in objectives]].values
    is_pareto = np.ones(len(vals), dtype=bool)
    for i in range(len(vals)):
        if not is_pareto[i]:
            continue
        for j in range(len(vals)):
            if i == j:
                continue
            dominates, strictly = True, False
            for k, (_, mode) in enumerate(objectives):
                better = vals[j, k] > vals[i, k] if mode == "MAX" else vals[j, k] < vals[i, k]
                worse  = vals[j, k] < vals[i, k] if mode == "MAX" else vals[j, k] > vals[i, k]
                if worse:
                    dominates = False
                    break
                if better:
                    strictly = True
            if dominates and strictly:
                is_pareto[i] = False
                break
    return df[is_pareto]

pareto_df = pareto_front(all_data, [
    ("Stokes Shift",    "MAX"),
    ("Quantum Yield",   "MAX"),
    ("Absorption Edge", "MIN"),
])

all_data["Pareto"]   = all_data.index.isin(pareto_df.index)
all_data["pt_size"]  = np.where(all_data["Pareto"], 10, 4)

print(f"Pareto-optimal: {len(pareto_df)} / {len(all_data)} experiments\n")
print("Pareto-optimal conditions:")
display(pareto_df[["Capping Ligand", "A-Site Cation", "Mn2+ Doping",
                   "Stokes Shift", "Quantum Yield", "Absorption Edge"]]
        .reset_index(drop=True)
        .sort_values("Absorption Edge")
        .style
        .set_caption("Pareto front — conditions no other experiment dominates on all 3 objectives")
        .background_gradient(subset=["Quantum Yield"], cmap="Greens")
        .background_gradient(subset=["Stokes Shift"],  cmap="Blues")
        .background_gradient(subset=["Absorption Edge"], cmap="Oranges_r")
        .format({"Mn2+ Doping": "{:.2f}", "Quantum Yield": "{:.0%}",
                 "Stokes Shift": "{:.0f} nm", "Absorption Edge": "{:.0f} nm"}))

fig = px.scatter_3d(
    all_data,
    x="Absorption Edge", y="Stokes Shift", z="Quantum Yield",
    color="Pareto",
    color_discrete_map={True: "crimson", False: "lightgray"},
    symbol="Pareto", symbol_map={True: "diamond", False: "circle"},
    size="pt_size", opacity=0.85,
    title="Multi-Objective Pareto Front — QD Synthesis Campaign (30 experiments)",
    labels={
        "Absorption Edge": "Absorption Edge ↓ (nm)",
        "Stokes Shift":    "Stokes Shift ↑ (nm)",
        "Quantum Yield":   "Quantum Yield ↑",
    },
    hover_data=["Capping Ligand", "A-Site Cation", "Mn2+ Doping", "X Composition"],
)
fig.update_layout(
    margin=dict(l=0, r=0, b=0, t=50),
    legend_title_text="Pareto Front",
    scene=dict(
        xaxis_title="Absorption Edge ↓ (nm)",
        yaxis_title="Stokes Shift ↑ (nm)",
        zaxis_title="QY ↑",
        bgcolor="#fafbfc",
    ),
    font=dict(family="system-ui, sans-serif"),
)
fig.show()

Pareto-optimal: 6 / 30 experiments

Pareto-optimal conditions:


,Capping Ligand,A-Site Cation,Mn2+ Doping,Stokes Shift,Quantum Yield,Absorption Edge
4,Oleylamine,Cs,3.10,203 nm,83%,395 nm
5,Oleylamine,Cs,2.79,207 nm,88%,397 nm
3,Oleylamine,Cs,3.32,209 nm,92%,403 nm
2,Oleylamine,Cs,2.11,208 nm,94%,411 nm
1,Oleylamine,Cs,2.11,211 nm,90%,415 nm
0,APTES,PEA,37.71,212 nm,89%,438 nm


---
## 6 · Tradeoffs and Limitations

### What the model does well

- **Mixed-space navigation.** 3×3 categorical × 3 continuous = effectively
  12-dimensional after one-hot encoding. Full factorial is intractable;
  Bayesian optimization finds the Pareto front in 30 experiments.
- **Non-obvious interactions.** APTES + MA cation + low Mn doping (1–10 mol%)
  holds 3 of 6 Pareto-optimal experiments — a cluster the LHS sampled only
  sparsely. The algorithm identified it within rounds 2–3.
- **Batch-efficient.** qLogNParEGO scores candidates jointly across a batch of 5,
  accounting for the information value of running them together rather than
  treating each independently.

### Where it breaks

- **Measurement data is illustrative.** In a real campaign, synthesis runs
  happen between each `recommend()` call. Fill in the form cells above with
  your actual measured values.
- **GP cold-start.** 15 points across 5 parameters is thin. Rounds 1–2
  recommendations are noisier than rounds 3–4. A 20–25 point LHS would
  reduce cold-start noise at modest experimental cost.
- **No feasibility constraints.** Ligand/cation incompatibilities and
  solubility limits are not encoded. Check recommended conditions for
  physical feasibility before synthesis.
- **Categorical encoding cost.** BayBE one-hot encodes categoricals, making
  the effective space 12-dimensional rather than 5. This inflates the
  cold-start problem but becomes manageable by round 3.
- **Compute cost.** qLogNParEGO is expensive for large Pareto fronts or
  more than 3 objectives. For scaled campaigns (>5 objectives or batch>10),
  consider q-NParEGO (random scalarization, faster, similar performance).